In [13]:
import pandas as pd
import plotly.express as px

In [14]:
cometa_df = pd.read_excel('Archivos/C_2006_1P.xlsx', sheet_name='Hoja3')
cometa_df

,Date (UT),Magn
0,7C2002 1 13.07265,20.5 P
1,7C2002 1 13.07348,P
2,7C2002 1 13.07597,P
3,7C2005 1 18.45625,19 P
4,7C2005 1 18.45707,P
...,...,...
4260,C2013 1 6.3737,16.6 P
4261,C2013 1 6.37912,16.7 P
4262,C2013 1 6.39325,16.7 P
4263,`C2012 12 29.72922,18.6 P


In [15]:
filas,columnas = cometa_df.shape
print(f'Registros: {filas}\nVariables: {columnas}')

Registros: 4265
Variables: 2


In [16]:
cometa_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4265 entries, 0 to 4264
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Date (UT)  4265 non-null   object
 1   Magn       4265 non-null   object
dtypes: object(2)
memory usage: 66.8+ KB


In [17]:
cometa_df.columns = cometa_df.columns.str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')
cometa_df.sample(10) 

,date_ut,magn
2263,C2013 1 4.91605,11.8 P
1175,C2012 11 8.72952,14.3 P
1410,C2013 1 3.92215,14.3 P
238,KC2012 12 7.7363,14.4 P
2480,C2013 1 6.02149,15.4 P
2801,KC2013 1 2.96091,15.5 P
3363,C2012 2 11.65252,18.8 P
3821,C2012 11 5.75495,14.2 P
1635,C2012 9 28.74617,14.3 P
3271,C2013 1 4.01559,16.4 P


In [18]:
cometa_df.magn = cometa_df.magn.str.extract(r'(\d+\.\d+)')
cometa_df.sample(10)

,date_ut,magn
621,C2012 12 10.21823,16.5
1640,C2012 9 29.73083,14.5
1556,C2012 9 17.70156,14.5
928,C2013 1 3.80759,16.5
2609,C2013 1 4.82197,14.9
1670,C2012 9 29.7655,14.4
3057,KC2012 12 6.74112,17.5
965,`C2012 12 28.73853,17.1
3592,C2012 12 11.87361,16.6
3023,|C2012 11 20.30674,18.7


In [19]:
cometa_df['magn'] = pd.to_numeric(cometa_df.magn)
cometa_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4265 entries, 0 to 4264
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   date_ut  4265 non-null   object 
 1   magn     3655 non-null   float64
dtypes: float64(1), object(1)
memory usage: 66.8+ KB


In [20]:
cometa_filtrado_df = cometa_df #[cometa_df.magn > -60].copy()
cometa_filtrado_df

,date_ut,magn
0,7C2002 1 13.07265,20.5
1,7C2002 1 13.07348,NaN
2,7C2002 1 13.07597,NaN
3,7C2005 1 18.45625,NaN
4,7C2005 1 18.45707,NaN
...,...,...
4260,C2013 1 6.3737,16.6
4261,C2013 1 6.37912,16.7
4262,C2013 1 6.39325,16.7
4263,`C2012 12 29.72922,18.6


In [21]:
cometa_dias_df = cometa_filtrado_df.date_ut.str.extract(r'(\d+\.\d+)')
cometa_dias_df = pd.to_numeric(cometa_dias_df[0])

cometa_procesado_df = pd.DataFrame()
cometa_procesado_df['obs_date'] = cometa_filtrado_df.date_ut.str.extract(r'(\d+ \d+)') 
cometa_procesado_df.obs_date = cometa_procesado_df.obs_date.str.replace(' ', '-')
cometa_procesado_df.obs_date = cometa_procesado_df.obs_date  + '-' + cometa_dias_df.apply(lambda dato: str(int(dato)))
cometa_procesado_df.obs_date = pd.to_datetime(pd.to_datetime(cometa_procesado_df.obs_date).dt.date)

cometa_procesado_df['magnitude'] = cometa_filtrado_df.magn
cometa_procesado_df.reset_index(inplace = True, drop= True)

cometa_procesado_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4265 entries, 0 to 4264
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   obs_date   4265 non-null   datetime64[ns]
 1   magnitude  3655 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 66.8 KB


In [22]:
cometa_procesado_df

,obs_date,magnitude
0,2002-01-13,20.5
1,2002-01-13,NaN
2,2002-01-13,NaN
3,2005-01-18,NaN
4,2005-01-18,NaN
...,...,...
4260,2013-01-06,16.6
4261,2013-01-06,16.7
4262,2013-01-06,16.7
4263,2012-12-29,18.6


In [23]:
fig = px.scatter(cometa_procesado_df, x='obs_date', y='magnitude', template= 'plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.show()

In [24]:
cometa_procesado_df.to_parquet('Archivos/12P_Procesado_MPC.parquet', index = False)

In [25]:
import requests
import pandas as pd

response = requests.get("https://data.minorplanetcenter.net/api/get-obs", json={"desigs": [r"2525"], "output_format":["OBS_DF"]})

if response.ok:
    mpc_obs_crude_df = pd.DataFrame(response.json()[0]['OBS_DF'])
    mpc_obs_crude_df

else:
    print("Error: ", response.status_code, response.content)

In [26]:
mpc_obs_proceed_df = pd.DataFrame()
mpc_obs_proceed_df['obs_date'] = mpc_obs_crude_df.obs80.str[15:19] + '-' + mpc_obs_crude_df.obs80.str[20:22] + '-' + mpc_obs_crude_df.obs80.str[23:25]
mpc_obs_proceed_df['magnitude'] = mpc_obs_crude_df.obs80.str[65:70].str.replace(' ', '')

mpc_obs_proceed_df.obs_date = pd.to_datetime(mpc_obs_proceed_df.obs_date)
mpc_obs_proceed_df.magnitude = pd.to_numeric(mpc_obs_proceed_df.magnitude)

mpc_obs_proceed_df


,obs_date,magnitude
0,1931-12-05,13.50
1,1931-12-05,NaN
2,1931-12-08,NaN
3,1936-09-13,12.90
4,1939-02-15,NaN
...,...,...
9439,2025-05-24,16.14
9440,2025-05-24,16.47
9441,2025-05-24,16.43
9442,2025-05-24,16.40


In [27]:
mpc_obs_proceed_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9444 entries, 0 to 9443
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   obs_date   9444 non-null   datetime64[ns]
 1   magnitude  9123 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 147.7 KB


In [28]:
# mpc_obs_filtered_df.magnitude.plot(kind='box')
px.box(mpc_obs_proceed_df, y='magnitude', template= 'plotly_dark')

In [29]:
q1 = mpc_obs_proceed_df['magnitude'].quantile(0.25)
q3 = mpc_obs_proceed_df['magnitude'].quantile(0.75)

IQR = q3 - q1 

limite_inferior = q1 - 1.5 * IQR
limite_superior = q3 + 1.5 * IQR

# Filtrar los outliers
mpc_obs_filtered_df = mpc_obs_proceed_df[(mpc_obs_proceed_df['magnitude'] >= limite_inferior) & (mpc_obs_proceed_df['magnitude'] <= limite_superior)]

px.box(mpc_obs_filtered_df, y='magnitude', template= 'plotly_dark')

In [30]:
fig = px.scatter(mpc_obs_filtered_df, x='obs_date', y='magnitude', template= 'plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.show()

In [31]:
mpc_obs_filtered_df.to_parquet('Archivos/2525_OStein_MPC.parquet', index = False)